# Black-Box Optimisation (BBO) Capstone Project — Interactive Walkthrough
**Imperial College London — Machine Learning & AI Capstone (Modules 12–25)**

This notebook provides a self-contained, reproducible walkthrough of our **Adaptive Multi-Surrogate Bayesian Optimisation with Automatic Relevance Determination (`BBO-ARD-MSB`)** framework across all 8 benchmark functions (2D to 8D).

### Non-Technical Overview (General Audience Summary)
> Imagine trying to find the highest peak across eight invisible mountain ranges while blindfolded, allowed only one altitude measurement per mountain each week. This project solves that challenge—known as Black-Box Optimisation—where testing a single setting in engineering, drug discovery, or artificial intelligence tuning is slow and expensive. Across twelve weekly rounds, our system combined statistical maps called Gaussian Processes with machine learning models to predict both where the highest peaks sit and where unexplored blind spots remain. By automatically identifying which control dials mattered most, our pipeline achieved major breakthroughs across the benchmarks, including a massive 8,662.48 peak on Function 5 and three consecutive all-time records in the final round.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Import core pipeline utilities from our unified production script (bbo.py)
from bbo import (
    load_function_data,
    define_candidate_models,
    evaluate_models_cv,
    generate_candidate_points_hybrid,
    filter_candidates_by_distance,
    expected_improvement,
    upper_confidence_bound,
    format_query_string
)

print("Loaded BBO Unified Engine successfully.")
print("Total functions configured: 8 (Dimensions: 2D to 8D)")

Loaded BBO Unified Engine successfully.
Total functions configured: 8 (Dimensions: 2D to 8D)


--- 
## 1. Dataset Composition & Multi-Week Progression (Weeks 1 to 13)
Each function folder (`function_1/` through `function_8/`) stores binary NumPy arrays (`initial_inputs.npy` and `initial_outputs.npy`) constrained to the unit hypercube `[0.0, 1.0]^d`. Over 12 sequential evaluation cycles, the dataset expanded from 175 initial points to **271 total evaluations**.

In [2]:
# Inspect all 8 function datasets through Week 13
dataset_rows = []
for fid in range(1, 9):
    X, y = load_function_data(fid)
    best_idx = int(np.argmax(y))
    dataset_rows.append({
        "Function": f"Func {fid}",
        "Dim": X.shape[1],
        "Total_Samples": len(y),
        "Min_Output": round(float(np.min(y)), 4),
        "Max_Output_W13": round(float(np.max(y)), 6),
        "Best_Coordinate_Norm": round(float(np.linalg.norm(X[best_idx])), 6)
    })

df_overview = pd.DataFrame(dataset_rows)
df_overview

   Function  Dim  Total_Samples  Min_Output   Max_Output_W13  Best_Coordinate_Norm
0    Func 1    2             22     -0.0114         0.000000              1.035234
1    Func 2    2             22     -0.0659         0.676790              1.087992
2    Func 3    3             27     -0.3980        -0.004807              0.880512
3    Func 4    4             42    -32.6259         0.367529              0.788412
4    Func 5    4             32      0.1129      8662.482500              2.000000
5    Func 6    5             32     -2.5712        -0.089540              1.216510
6    Func 7    6             42     -0.0787         3.227650              1.158204
7    Func 8    8             52      5.5922         9.981970              1.102189

---
## 2. Surrogate Model Benchmarking & ARD Lengthscale Learning
For every function, `evaluate_models_cv` conducts a 5-fold cross-validation tournament across 8 candidate models:
- **Anisotropic Gaussian Processes (ARD)**: `Matérn 2.5 ARD`, `Matérn 1.5 ARD`, and `RBF ARD` with dimension-specific lengthscales $\mathbf{\Theta} = [\ell_1, \dots, \ell_d]$.
- **Deterministic Baselines**: `Extra Trees`, `Random Forest`, `Gradient Boosting`, `Polynomial Ridge` (degree 2), and `MLPRegressor` `(64, 32)`.

We dynamically select the best-performing Gaussian Process kernel per function to ensure calibrated posterior mean $\mu(x)$ and standard deviation $\sigma(x)$ during acquisition.

In [3]:
# Load pre-computed Week 13 5-Fold Cross-Validation & Acquisition Summary
with open("week13_summary.json", "r") as f:
    w13_summary = json.load(f)

summary_table = []
for item in w13_summary:
    top_cv = item["cv_results"][0]
    summary_table.append({
        "Function": f"Func {item['func_id']}",
        "Dim": item["dim"],
        "Winning_Overall": item["best_model"],
        "Selected_GP_Kernel": item["chosen_gp_surrogate"],
        "CV_MSE": round(top_cv["CV_MSE_Mean"], 6),
        "CV_R2": round(top_cv["CV_R2_Mean"], 3),
        "Current_Best_y": round(item["current_best_y"], 6),
        "Final_Week13_Query": item["submission_string"]
    })

pd.DataFrame(summary_table)

   Function  Dim         Winning_Overall         Selected_GP_Kernel    CV_MSE  CV_R2  Current_Best_y               Final_Week13_Query
0    Func 1    2     GP (Matern 2.5 ARD)        GP (Matern 2.5 ARD)  0.000001 -0.050        0.000000               0.175879-0.814070
1    Func 2    2       Gradient Boosting        GP (Matern 1.5 ARD)  0.019380  0.495        0.676790               0.693467-0.849246
2    Func 3    3     GP (Matern 1.5 ARD)        GP (Matern 1.5 ARD)  0.006142  0.291       -0.004807        0.997007-0.530232-0.396284
3    Func 4    4     GP (Matern 1.5 ARD)        GP (Matern 1.5 ARD)  2.735104  0.946        0.367529 0.388793-0.445325-0.366668-0.367742
4    Func 5    4             Extra Trees        GP (Matern 1.5 ARD) 489120.41  0.884     8662.482500 0.103158-0.023194-0.994812-0.955309
5    Func 6    5     GP (Matern 1.5 ARD)        GP (Matern 1.5 ARD)  0.068419  0.788       -0.089540 0.524565-0.431072-0.589917-0.811460-0.161023
6    Func 7    6            GP (RBF ARD)     

---
## 3. Visualising the 13-Week Progression & Diagnostic Landscapes
Our pipeline automatically generates diagnostic charts in `visualizations/`, including the 4-panel master dashboard (`weekly_progress_summary.png`) and individual 4-panel function sensitivity slices (`function_1_week13.png` to `function_8_week13.png`).

In [4]:
print("Displaying Master Weekly Progress Dashboard and High-Dimensional ARD Slice (Function 8):")
if os.path.exists("visualizations/weekly_progress_summary.png"):
    display(Image(filename="visualizations/weekly_progress_summary.png", width=900))
if os.path.exists("visualizations/function_8_week13.png"):
    display(Image(filename="visualizations/function_8_week13.png", width=900))

Displaying Master Weekly Progress Dashboard and High-Dimensional ARD Slice (Function 8):
